<a href="https://colab.research.google.com/github/AndrewRaj0510/ClaimLens/blob/main/training/eval_cardd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClaimLens: detection and cost-estimate eval on the CarDD test split

Runs `services/eval`'s **detection** suite where CarDD is (your Drive), because CarDD can't be put in
the repo or in CI. It measures the deployed `best.onnx` through the services' own code:

- **Detection:** box and mask mAP@0.5 and mAP@0.5:0.95 (COCO-style, checked against pycocotools in
  `services/eval/test_metrics.py`), per class, mean IoU of the matches, and precision/recall at the
  confidence the agent really uses. The vision service's own ONNX post-processing, not Ultralytics.
- **Cost error:** each test photo as a claim. Predicted cost = the production path (vision service →
  agent's severity rules → pricing server); "true" cost = the same rules on the labelled masks. So it's
  the error the vision pipeline adds, per damage class.

**Runtime:** CPU is enough (about 0.25 s per image on a laptop CPU, so a few minutes for the 374 test images). No GPU needed.

**Output:** a summary JSON (numbers only, no images or file names), saved to Drive. Copy it into the repo
as `services/eval/results/detection.json` and commit it; CI gates it against `services/eval/baseline.json`.

**Data licence:** CarDD is for research and education only. Nothing from CarDD leaves Drive except
these summary numbers.

## 1. Config

In [1]:
# Edit these to match your Drive (the same values as train_cardd.ipynb).
CARDD_ZIP = (
    ""  # the zip's full path, or leave empty to find a .zip with "cardd" in its name
)
CARDD_ROOT = ""  # an already-unzipped folder (the one holding annotations/)
RUNS_DIR = "/content/drive/MyDrive/claimlens/runs"
RUN_NAME = "cardd_yolo11s_seg"
REPO_URL = "https://github.com/AndrewRaj0510/ClaimLens.git"
DATASET_DIR = "/content/cardd_yolo"

# The ONNX model the vision service runs (exported by train_cardd.ipynb).
MODEL_PATH = f"{RUNS_DIR}/{RUN_NAME}/weights/best.onnx"
OUT_PATH = f"{RUNS_DIR}/eval/detection.json"

## 2. Mount Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Unzip CarDD

Copies the zip from Drive to Colab's own disk and unzips it there (reading from the
copy is much faster than from Drive). Takes a few minutes. After a disconnect it runs
again, because Colab's disk is wiped; your training progress on Drive is not.

In [ ]:
import glob
import shutil
import zipfile
from pathlib import Path

DRIVE = "/content/drive/MyDrive"
EXTRACT_DIR = Path("/content/CarDD")


def find_cardd_root(base):
    """The folder that holds annotations/instances_train2017.json, wherever the zip put it."""
    hits = [
        p
        for p in Path(base).rglob("instances_train2017.json")
        if p.parent.name == "annotations" and "__MACOSX" not in p.parts
    ]
    return str(min(hits).parent.parent) if hits else None


if CARDD_ROOT and Path(CARDD_ROOT, "annotations").is_dir():
    print("Using the unzipped CarDD folder:", CARDD_ROOT)
else:
    CARDD_ROOT = find_cardd_root(EXTRACT_DIR) if EXTRACT_DIR.exists() else None
    if CARDD_ROOT:
        print("Already unzipped in this session.")
    else:
        zip_path = CARDD_ZIP
        if not zip_path:
            zips = sorted(
                {p for d in ("", "*/", "*/*/") for p in glob.glob(f"{DRIVE}/{d}*.zip")}
            )
            print("Zip files in your Drive (3 folders deep):", *zips, sep="\n  ")
            matches = [z for z in zips if "cardd" in Path(z).name.lower()]
            assert len(matches) == 1, (
                "Couldn't pick the CarDD zip automatically. Copy its path from the list "
                "above into CARDD_ZIP in the config cell, run that cell, then this one."
            )
            zip_path = matches[0]
        local_zip = Path("/content/CarDD.zip")
        print(f"Copying {zip_path} to Colab's disk...")
        shutil.copyfile(zip_path, local_zip)
        print("Unzipping...")
        with zipfile.ZipFile(local_zip) as z:
            z.extractall(EXTRACT_DIR)
        local_zip.unlink()  # free the disk space
        CARDD_ROOT = find_cardd_root(EXTRACT_DIR)

assert CARDD_ROOT, (
    "Unzipped, but there's no annotations/instances_train2017.json inside. "
    "Run  !ls -R /content/CarDD | head -60  in a new cell and send the output."
)
print("CARDD_ROOT =", CARDD_ROOT)

## 4. Fetch the repo and install the eval's requirements

Pulls `services/eval` from GitHub, so push your latest changes first. The eval imports the code it
measures (vision post-processing, agent severity rules, pricing) from the same checkout.

In [ ]:
import os
import sys

if not os.path.exists("/content/ClaimLens"):
    !git clone --depth 1 {REPO_URL} /content/ClaimLens
else:
    !git -C /content/ClaimLens pull --ff-only
%pip install -q -r /content/ClaimLens/services/eval/requirements.txt
sys.path.insert(0, "/content/ClaimLens/training")

## 5. Convert CarDD → YOLO-seg (same converter as training; the test split is 374 images)

In [ ]:
from convert_cardd import convert

names, stats = convert(CARDD_ROOT, DATASET_DIR)
print("classes:", names)
print("test:", dict(stats["test"]))
assert stats["test"]["images"] == 374, (
    "expected the official CarDD test split (374 images)"
)

## 6. Run the eval

In [ ]:
from pathlib import Path

assert Path(MODEL_PATH).exists(), (
    f"no model at {MODEL_PATH}: run train_cardd.ipynb's export cell"
)
Path(OUT_PATH).parent.mkdir(parents=True, exist_ok=True)
!cd /content/ClaimLens/services/eval && python run.py detection --data {DATASET_DIR} --model "{MODEL_PATH}" --out "{OUT_PATH}"

## 7. Summary

In [ ]:
import json

record = json.loads(Path(OUT_PATH).read_text())
r = record["result"]
print(
    f"images {r['images']}, model sha256 {r['model_sha256'][:12]}..., operating confidence {r['operating_confidence']}"
)
print(
    f"{'':16s} {'box AP50':>9s} {'box AP':>7s} {'mask AP50':>9s} {'mask AP':>8s} {'P@op':>6s} {'R@op':>6s}"
)
print(
    f"{'all':16s} {r['box']['map50']:9.3f} {r['box']['map50_95']:7.3f} {r['mask']['map50']:9.3f} {r['mask']['map50_95']:8.3f}"
)
for name, m in r["mask"]["per_class"].items():
    if not m["n_gt"]:
        continue  # no labelled instances of this class in the split
    b = r["box"]["per_class"][name]
    print(
        f"{name:16s} {b['ap50']:9.3f} {b['ap50_95']:7.3f} {m['ap50']:9.3f} {m['ap50_95']:8.3f} {m['precision'] or 0:6.2f} {m['recall']:6.2f}"
    )
c = r["cost"]
print()
print(
    f"cost: MAE ${c['mae']:,.0f} on a mean true cost of ${c['mean_true_cost']:,.0f}; bias ${c['bias']:,.0f};"
    f" median APE {c['median_ape']:.1%}; within 20% {c['within_20pct']:.1%};"
    f" true cost in quoted range {c['true_cost_in_quoted_range']:.1%}"
)
for name, p in c["per_class"].items():
    print(
        f"  {name:14s} photos {p['photos_with_it']:3d}  missed {p['missed_rate']}  severity agreement {p['severity_agreement']}"
        f"  spurious {p['spurious']:3d}  share of $ error {p['share_of_error']:.0%}"
    )

## Next

1. Download `detection.json` from `RUNS_DIR/eval/` and save it in the repo as
   `services/eval/results/detection.json`.
2. Re-baseline the detection gates in `services/eval/baseline.json` from these numbers (they're
   provisional until this first run), and add cost gates (e.g. `cost.median_ape`).
3. Run `python services/eval/run.py gate --require detection`, then commit both files.
4. Record the numbers in `docs/metrics.md` next to Ultralytics' Phase 1 numbers.